In [1]:
import pandas as pd 
import numpy as np

import sys
import os
EDHREC_REPO_PATH = '..'
sys.path.append(os.path.abspath(EDHREC_REPO_PATH))
# Set the environment variables
os.environ['APP_ENVIRONMENT'] = 'dev'
os.environ['APP_NAME'] = 'api'
os.environ['AWS_DEFAULT_REGION'] = 'us-east-2'
os.environ['APP_CONFIG_DIR'] = os.path.abspath(f'{EDHREC_REPO_PATH}/edhrec/configs/')

from edhrec.plus.query import athena
from edhrec.plus.jobs.count_warehouse.config import (
    ATHENA_WORKGROUP,
    ICEBERG_WAREHOUSE,
)

import plotly.express as px
import logging

# Add this at the top of your notebook/script
logging.getLogger('PIL').setLevel(logging.WARNING)

### Querying from Iceberg to get "depth" data

In [2]:
#engine = athena.AthenaQueryEngine(athena_workgroup = ATHENA_WORKGROUP, iceberg_warehouse = ICEBERG_WAREHOUSE)

# ending_columns = ['commander', 'commander2', 'color', 'earliest_sets', 'released_at', 'user_id', 'first_create_time', 'first_update_date', 'last_update_date', 'total_active_time', 'num_updates']

# query_str = f"""

# WITH date_span AS (
#     SELECT MIN(update_time) as min_update_time, MAX(update_time) as max_update_time
#     FROM prod.all_decks
#     WHERE user_id IS NOT NULL and create_time IS NOT NULL
# ),

# deck_history AS (
#     SELECT commander_id, commander2_id, deck_color_id, user_id, create_time, DATE(update_time) as update_date
#     FROM prod.all_decks
#     WHERE user_id IS NOT NULL and update_time >= create_time + INTERVAL '60' DAY
# ),

# -- limit to only one update per commander/user/day to avoid overcounting updates for users who make multiple updates in a day
# deck_history_daily_updates AS (
#     SELECT commander_id, commander2_id, deck_color_id, user_id, update_date, MIN(create_time) as create_time, 1 as num_updates
#     FROM deck_history
#     GROUP BY commander_id, commander2_id, deck_color_id, user_id, update_date
# ),

# -- get total number of updates by user and commander
# user_commander_update_counts AS (
#     SELECT commander_id, commander2_id, user_id, 
#         MIN(create_time) as first_create_time,
#         MIN(update_date) as first_update_date,
#         MAX(update_date) as last_update_date,
#         DATE_DIFF('day', MIN(create_time), MAX(update_date)) as total_active_time,
#         SUM(num_updates) as num_updates
#     FROM deck_history_daily_updates
#     GROUP BY commander_id, commander2_id, user_id
# ),

# commander_list AS (
#     SELECT DISTINCT commander_id, commander2_id, deck_color_id
#     FROM deck_history
# ), 

# commander_info AS (
#     SELECT commander_id, commander2_id, 
#         cr1.name as commander, cr2.name as commander2,
#         GREATEST(cr1.released_at, COALESCE(cr2.released_at, cr1.released_at)) as released_at,
#         CASE WHEN cr2.earliest_set IS NULL 
#             THEN ARRAY[cr1.earliest_set]
#             ELSE ARRAY[cr1.earliest_set, cr2.earliest_set]
#             END as earliest_sets,
#         cc.alphabetical_combination as color
#     FROM commander_list
#     JOIN prod.card_ref cr1 ON commander_list.commander_id = cr1.card_id
#     LEFT JOIN prod.card_ref cr2 ON commander_list.commander2_id = cr2.card_id
#     JOIN prod.color_combinations cc ON commander_list.deck_color_id = cc.value
# )

# SELECT {', '.join(ending_columns)}
# FROM user_commander_update_counts ucuc
# JOIN commander_info ci
#     ON ucuc.commander_id = ci.commander_id
#     AND ucuc.commander2_id IS NOT DISTINCT FROM ci.commander2_id
# ORDER BY commander, commander2, user_id
# """

# raw_results = engine.execute(query_str)

# df = pd.DataFrame(raw_results, columns=ending_columns)

# df


In [9]:
df = pd.read_csv("depth_vs_breadth_raw_data.csv")
df['released_at'] = pd.to_datetime(df['released_at'])

df['time_since_release'] = (pd.to_datetime('today') - pd.to_datetime(df['released_at'])).dt.days
df['time_since_release'] = np.where(df['time_since_release'] >= 365*2, 365*2, df['time_since_release'])

# convert from string representation of list back to list
df['earliest_sets'] = df['earliest_sets'].apply(lambda x: tuple(eval(x)) if isinstance(x, str) else x)

df

/var/folders/_k/kyjp44cj6493znjb_xjbld0r0000gn/T/ipykernel_42880/1385567191.py:1: DtypeWarning: Columns (0: commander2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("depth_vs_breadth_raw_data.csv")


,commander,commander2,color,earliest_sets,released_at,user_id,first_create_time,first_update_date,last_update_date,total_active_time,num_updates,time_since_release
0,"""Brims"" Barone, Midway Mobster",NaN,BW,"(unf,)",2022-10-07,156055,2023-04-29 15:06:53+00:00,2025-12-14,2025-12-14,959,1,730
1,"""Brims"" Barone, Midway Mobster",NaN,BW,"(unf,)",2022-10-07,19915,2023-11-19 00:00:00+00:00,2024-03-24,2024-03-24,126,1,730
2,"""Brims"" Barone, Midway Mobster",NaN,BW,"(unf,)",2022-10-07,201498,2025-10-08 06:49:08+00:00,2025-12-22,2025-12-26,78,2,730
3,"""Brims"" Barone, Midway Mobster",NaN,BW,"(unf,)",2022-10-07,236905,2024-09-18 00:00:00+00:00,2024-12-23,2024-12-23,96,1,730
4,"""Brims"" Barone, Midway Mobster",NaN,BW,"(unf,)",2022-10-07,35613,2025-03-15 00:00:00+00:00,2025-06-05,2025-06-05,82,1,730
...,...,...,...,...,...,...,...,...,...,...,...,...
2662348,"Éowyn, Shieldmaiden",NaN,RUW,"(ltc,)",2023-06-23,zootfarm,2024-12-01 00:00:00+00:00,2025-05-09,2025-11-22,356,5,730
2662349,"Éowyn, Shieldmaiden",NaN,RUW,"(ltc,)",2023-06-23,zozpreston,2023-08-25 16:25:52.807000+00:00,2023-10-28,2023-10-28,63,1,730
2662350,"Éowyn, Shieldmaiden",NaN,RUW,"(ltc,)",2023-06-23,zulla24,2025-01-27 00:00:00+00:00,2025-07-08,2026-01-25,363,3,730
2662351,"Éowyn, Shieldmaiden",NaN,RUW,"(ltc,)",2023-06-23,zunigarillo,2025-10-25 05:09:30.287000+00:00,2025-12-31,2026-04-06,162,7,730


In [10]:
df['commanders'] = df.apply(lambda row: f"{row['commander']} + {row['commander2']}" if pd.notna(row['commander2']) else row['commander'], axis=1)

keep_commanders = df.groupby('commanders')['user_id'].nunique().reset_index()\
    .rename(columns={"user_id": "num_users"})\
    .query("num_users > 200")\
    ['commanders'].values.tolist()

keep_commanders

['Aang, at the Crossroads',
 'Aatchik, Emerald Radian',
 'Abaddon the Despoiler',
 "Abdel Adrian, Gorion's Ward + Candlekeep Sage",
 "Abdel Adrian, Gorion's Ward + Far Traveler",
 'Abigale, Eloquent First-Year',
 'Abomination of Llanowar',
 'Absolute Virtue',
 'Abuelo, Ancestral Echo',
 'Acererak the Archlich',
 'Aclazotz, Deepest Betrayal',
 'Adeline, Resplendent Cathar',
 'Adeliz, the Cinder Wind',
 'Admiral Beckett Brass',
 'Admiral Brass, Unsinkable',
 'Adrix and Nev, Twincasters',
 'Aegar, the Freezing Flame',
 'Aerith Gainsborough',
 'Aerith, Last Ancient',
 'Aesi, Tyrant of Gyre Strait',
 'Aeve, Progenitor Ooze',
 'Agatha of the Vile Cauldron',
 'Agent Frank Horrigan',
 'Agrus Kos, Eternal Soldier',
 'Aisha of Sparks and Smoke',
 'Ajani, Nacatl Pariah',
 'Akiri, Fearless Voyager',
 'Akiri, Line-Slinger + Ardenn, Intrepid Archaeologist',
 'Akiri, Line-Slinger + Silas Renn, Seeker Adept',
 'Akiri, Line-Slinger + Thrasios, Triton Hero',
 'Akroma, Vision of Ixidor + Rograkh, Son of 

In [11]:
df_select = df.query("commanders in @keep_commanders")

df_summary = df_select.groupby(["commanders", "commander", "commander2", "released_at", "color", "earliest_sets", "time_since_release"], dropna = False).agg({
    'num_updates': ['mean'],
    'user_id': 'count'
}).reset_index()

df_summary.columns = ['commanders', 'commander', 'commander2', 'released_at', 'color', 'earliest_sets', 'time_since_release', 'average_num_updates', 'num_users']
df_summary

,commanders,commander,commander2,released_at,color,earliest_sets,time_since_release,average_num_updates,num_users
0,"Aang, at the Crossroads","Aang, at the Crossroads",NaN,2025-11-21,GUW,"(tla,)",159,3.291667,792
1,"Aatchik, Emerald Radian","Aatchik, Emerald Radian",NaN,2025-02-14,BG,"(dft,)",439,3.223819,487
2,Abaddon the Despoiler,Abaddon the Despoiler,NaN,2022-10-07,BRU,"(40k,)",730,2.567592,2197
3,"Abdel Adrian, Gorion's Ward + Candlekeep Sage","Abdel Adrian, Gorion's Ward",Candlekeep Sage,2022-06-10,UW,"(clb, clb)",730,3.223785,1564
4,"Abdel Adrian, Gorion's Ward + Far Traveler","Abdel Adrian, Gorion's Ward",Far Traveler,2022-06-10,W,"(clb, clb)",730,3.054118,425
...,...,...,...,...,...,...,...,...,...
1714,Zurgo Stormrender,Zurgo Stormrender,NaN,2025-04-11,BRW,"(tdc,)",383,4.844936,5746
1715,Zurgo and Ojutai,Zurgo and Ojutai,NaN,2023-04-21,RUW,"(mom,)",730,3.238671,662
1716,"Zurgo, Thunder's Decree","Zurgo, Thunder's Decree",NaN,2025-04-11,BRW,"(tdm,)",383,3.739242,1557
1717,"Zurzoth, Chaos Rider","Zurzoth, Chaos Rider",NaN,2020-07-17,R,"(jmp,)",730,2.953264,1348


In [12]:
# set to parent set name
set_to_parent_set = pd.read_csv("/Users/juliamaddalena/edhrec/edhrec-notebooks/wotc_presentation_analyses/set_to_parent_map.csv")

df_summary_with_sets = df_summary.explode(column = "earliest_sets")\
    .merge(set_to_parent_set, left_on='earliest_sets', right_on='code', how='left')\
    .groupby(["commanders", "commander", "commander2", "released_at", "time_since_release", "color"], dropna = False)\
        .agg({'average_num_updates': 'first', 'num_users': 'first', 
              'parent_set': lambda x: list(x.unique())})\
        .reset_index()

df_summary_with_sets

,commanders,commander,commander2,released_at,time_since_release,color,average_num_updates,num_users,parent_set
0,"Aang, at the Crossroads","Aang, at the Crossroads",NaN,2025-11-21,159,GUW,3.291667,792,[Avatar: The Last Airbender]
1,"Aatchik, Emerald Radian","Aatchik, Emerald Radian",NaN,2025-02-14,439,BG,3.223819,487,[Aetherdrift]
2,Abaddon the Despoiler,Abaddon the Despoiler,NaN,2022-10-07,730,BRU,2.567592,2197,"[Warhammer 40,000 Commander]"
3,"Abdel Adrian, Gorion's Ward + Candlekeep Sage","Abdel Adrian, Gorion's Ward",Candlekeep Sage,2022-06-10,730,UW,3.223785,1564,[Commander Legends: Battle for Baldur's Gate]
4,"Abdel Adrian, Gorion's Ward + Far Traveler","Abdel Adrian, Gorion's Ward",Far Traveler,2022-06-10,730,W,3.054118,425,[Commander Legends: Battle for Baldur's Gate]
...,...,...,...,...,...,...,...,...,...
1714,Zurgo Stormrender,Zurgo Stormrender,NaN,2025-04-11,383,BRW,4.844936,5746,[Tarkir: Dragonstorm]
1715,Zurgo and Ojutai,Zurgo and Ojutai,NaN,2023-04-21,730,RUW,3.238671,662,[March of the Machine]
1716,"Zurgo, Thunder's Decree","Zurgo, Thunder's Decree",NaN,2025-04-11,383,BRW,3.739242,1557,[Tarkir: Dragonstorm]
1717,"Zurzoth, Chaos Rider","Zurzoth, Chaos Rider",NaN,2020-07-17,730,R,2.953264,1348,[Jumpstart]


In [16]:
df_summary_with_sets.to_parquet("depth_vs_breadth_plot_df.parquet", index=False)

In [15]:
df_summary_with_sets

,commanders,commander,commander2,released_at,time_since_release,color,average_num_updates,num_users,parent_set
0,"Aang, at the Crossroads","Aang, at the Crossroads",NaN,2025-11-21,159,GUW,3.291667,792,[Avatar: The Last Airbender]
1,"Aatchik, Emerald Radian","Aatchik, Emerald Radian",NaN,2025-02-14,439,BG,3.223819,487,[Aetherdrift]
2,Abaddon the Despoiler,Abaddon the Despoiler,NaN,2022-10-07,730,BRU,2.567592,2197,"[Warhammer 40,000 Commander]"
3,"Abdel Adrian, Gorion's Ward + Candlekeep Sage","Abdel Adrian, Gorion's Ward",Candlekeep Sage,2022-06-10,730,UW,3.223785,1564,[Commander Legends: Battle for Baldur's Gate]
4,"Abdel Adrian, Gorion's Ward + Far Traveler","Abdel Adrian, Gorion's Ward",Far Traveler,2022-06-10,730,W,3.054118,425,[Commander Legends: Battle for Baldur's Gate]
...,...,...,...,...,...,...,...,...,...
1714,Zurgo Stormrender,Zurgo Stormrender,NaN,2025-04-11,383,BRW,4.844936,5746,[Tarkir: Dragonstorm]
1715,Zurgo and Ojutai,Zurgo and Ojutai,NaN,2023-04-21,730,RUW,3.238671,662,[March of the Machine]
1716,"Zurgo, Thunder's Decree","Zurgo, Thunder's Decree",NaN,2025-04-11,383,BRW,3.739242,1557,[Tarkir: Dragonstorm]
1717,"Zurzoth, Chaos Rider","Zurzoth, Chaos Rider",NaN,2020-07-17,730,R,2.953264,1348,[Jumpstart]


In [8]:
fig = px.scatter(
    df_summary_with_sets,
    x="num_users",
    y="average_num_updates",
    color="time_since_release",
    color_continuous_scale=px.colors.sequential.thermal[::-1],
    hover_name="commanders",
    hover_data={
        "num_users": ":,.0f",
        "average_num_updates": ":.3f",
        "commanders": False,
    },
    labels={
        "num_users": "# users with a deck for this commander",
        "average_num_updates": "average # deck updates for this commander across users",
        "time_since_release": "days since commander release  (2 year cap)",
    },
    title="Breadth (x) vs Depth (y) for commanders in the last two years",
    log_x=True,
    width=1200,
    height=800,
    render_mode="svg",
)

fig.write_html("breadth_vs_depth_updates_after_60_day_burn_in.html")
fig.show()